### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="lymphography",
    dataset_year="1988",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C54598",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/63/lymphography.zip && unzip lymphography.zip lymphography.data && rm lymphography.zip && mkdir -p local-data-warehouse/lymphography && mv lymphography.data local-data-warehouse/lymphography/
""",
    # References
    academic_reference_bibtex="""@misc{Zwitter1988Lymphography,
  author       = {Zwitter, M. and Soklic, M.},
  title        = {{Lymphography}},
  year         = {1988},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C54598}
}
""",
    academic_reference_bibtex_key="Zwitter1988Lymphography",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- As we only have a too small number of samples for 2 out of the 4 classes, we drop those and keep only the 2 classes with more samples, to have a binary classification task. Thus, we aim to predict given the data if it is metastases or malign lymph.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="class",
)

## Preprocessing

In [2]:
import pandas as pd

columns = ["class","lymphatics","block_of_affere","block_of_lymph_c","block_of_lymph_s","by_pass","extravasates","regeneration_of","early_uptake_in","lym_nodes_dimin","lym_nodes_enlar","changes_in_lymph","defect_in_node","changes_in_node","changes_in_structure","special_forms","dislocation_of","exclusion_of_no","num_of_nodes_in"
]

df = pd.read_csv(dataset_mold.path / "lymphography.data", header=None, na_values="?", names=columns)
print("Loaded data shape:", df.shape)

df = df[df["class"].isin([2, 3])]
df["class"] = df["class"].map({2: "metastases", 3: "malign lymph"})


as_cat_type = ["class", "lymphatics","block_of_affere","block_of_lymph_c","block_of_lymph_s","by_pass","extravasates","regeneration_of","early_uptake_in", "changes_in_lymph","defect_in_node","changes_in_node","changes_in_structure","special_forms","dislocation_of","exclusion_of_no"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle data

Loaded data shape: (148, 19)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 142
Columns: 19
Use sampling: False (sample size: 142)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['num_of_nodes_in', 'changes_in_structure', 'defect_in_node', 'changes_in_node', 'lym_nodes_enlar', 'lymphatics', 'changes_in_lymph', 'special_forms', 'block_of_affere', 'block_of_lymph_c']
Rows remaining as candidates after top-10 filter: 4 (of 142)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,class,lymphatics,block_of_affere,block_of_lymph_c,block_of_lymph_s,by_pass,extravasates,regeneration_of,early_uptake_in,lym_nodes_dimin,lym_nodes_enlar,changes_in_lymph,defect_in_node,changes_in_node,changes_in_structure,special_forms,dislocation_of,exclusion_of_no,num_of_nodes_in
0,metastases,3,2,2,2,2,2,1,2,1,2,3,3,3,4,3,2,2,7
1,malign lymph,3,1,1,1,1,1,1,2,1,2,2,4,2,4,3,2,2,3
2,metastases,2,2,1,1,2,2,1,1,1,2,2,4,2,8,3,2,2,1
3,malign lymph,2,1,1,1,1,2,1,2,1,3,3,4,2,8,3,2,2,3
4,metastases,2,2,2,1,2,2,1,2,1,3,3,3,3,8,3,2,2,2


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,class,category,0.0,0.0,2.0,"metastases, malign lymph"
1,lymphatics,category,0.0,0.0,3.0,"2, 3, 4"
2,block_of_affere,category,0.0,0.0,2.0,"2, 1"
3,block_of_lymph_c,category,0.0,0.0,2.0,"1, 2"
4,block_of_lymph_s,category,0.0,0.0,2.0,"1, 2"
5,by_pass,category,0.0,0.0,2.0,"1, 2"
6,extravasates,category,0.0,0.0,2.0,"1, 2"
7,regeneration_of,category,0.0,0.0,2.0,"1, 2"
8,early_uptake_in,category,0.0,0.0,2.0,"2, 1"
9,changes_in_lymph,category,0.0,0.0,3.0,"2, 3, 1"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
lym_nodes_dimin,142.0,1.014085,0.118257,1.0,2.0
lym_nodes_enlar,142.0,2.521127,0.814046,1.0,4.0
num_of_nodes_in,142.0,2.535211,1.859085,1.0,8.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column               rank                            
block_of_affere      1                2     80  56.34
                     2                1     62  43.66
block_of_lymph_c     1                1    118  83.10
                     2                2     24  16.90
block_of_lymph_s     1                1    137  96.48
                     2                2      5   3.52
by_pass              1                1    110  77.46
                     2                2     32  22.54
changes_in_lymph     1                2     75  52.82
                     2                3     65  45.77
                     3                1      2   1.41
changes_in_node      1                3     75  52.82
                     2                2     39  27.46
                     3                4     25  17.61
                     4                1      3   2.11
changes_in_structure 1                8     44  30.99
                     2                4     30  21.13
                     3                5     26  18.31
                     4                3     19  13.38
                     5                2     13   9.15
class                1       metastases     81  57.04
                     2     malign lymph     61  42.96
defect_in_node       1                4     48  33.80
                     2                2     47  33.10
                     3                3     46  32.39
                     4                1      1   0.70
dislocation_of       1                2     96  67.61
                     2                1     46  32.39
early_uptake_in      1                2    102  71.83
                     2                1     40  28.17
exclusion_of_no      1                2    114  80.28
                     2                1     28  19.72
extravasates         1                1     72  50.70
                     2                2     70  49.30
lymphatics           1                2     67  47.18
                     2                3     42  29.58
                     3                4     33  23.24
regeneration_of      1                1    136  95.77
                     2                2      6   4.23
special_forms        1                3     74  52.11
                     2                2     43  30.28
                     3                1     25  17.61

In [8]:
# Target Distribution
target_df

,count,pct
class,,
metastases,81,57.04
malign lymph,61,42.96


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c75c5-725c-708c-9794-eccc46c0bf81
5d6400b72c006eaf0eebce997240f9d3373ce17aa0d4fd88b4818918629acf8c
